### Resolución de una serie de preguntas mediante la información de la capa Gold

**¿Qué clientes operan con un valor por encima del precio de mercado? ¿Con qué instrumentos?**

Revisamos los clientes pero acotamos en primera instancia solo a usar cedears y acciones, qué se operan en ARS. Con bonos soberanos habrá qué hacer la separación de cuáles se operan en dolares y cuáles en pesos.

Buscamos aquellos clientes qué tienen más de la mitad de sus transacciones del conjunto analizado a valores superiores al valor de mercado

In [0]:
%sql
WITH conjunto_a_analizar AS ( 
    -- Tomamos datos qué sean de los simbolos a analizar y qué tengan información de mercado
    SELECT * FROM iol_challenge.gold.fact_transaction 
        WHERE 
            simbolo_tipo IN ("Cedear", "Bono soberano") AND
            High IS NOT NULL AND Low is not null
), flagged_data AS (
    SELECT
        CASE WHEN valor_mercado_promedio > precio
            THEN 1
        ELSE 0
        END  AS transacion_superior_a_mercado
        ,*
    FROM conjunto_a_analizar
), data_agrupada_por_cliente AS (
    select 
        id_cliente
        ,CASE WHEN SUM(transacion_superior_a_mercado) > COUNT(*)/2
            THEN 1
            ELSE 0
        END AS cliente_con_transacciones_superiores_a_mercado
        FROM flagged_data GROUP BY id_cliente
) select id_cliente from data_agrupada_por_cliente WHERE cliente_con_transacciones_superiores_a_mercado=1

    

id_cliente
CLI57CF148E
CLIB2E038B2
CLICDCADF17
CLI5EB462C3
CLI66DACBC1
CLI2E1FC58C
CLIF7E028BD
CLI17C6C93A
CLI4FCE02E5
CLI92CC0774


Revisamos en que instrumentos tuvieron más de la mitad de sus operaciones a precio superior a mercado.

In [0]:
%sql
WITH conjunto_a_analizar AS ( 
    -- Tomamos datos qué sean de los simbolos a analizar y qué tengan información de mercado
    SELECT * FROM iol_challenge.gold.fact_transaction 
        WHERE 
            simbolo_tipo IN ("Cedear", "Bono soberano") AND
            High IS NOT NULL AND Low is not null
), flagged_data AS (
    SELECT
        CASE WHEN valor_mercado_promedio > precio
            THEN 1
        ELSE 0
        END  AS transacion_superior_a_mercado
        ,*
    FROM conjunto_a_analizar
), data_agrupada_por_instrumento AS (
    select 
        simbolo_titulo
        ,simbolo_tipo
        ,CASE WHEN SUM(transacion_superior_a_mercado) > COUNT(*)/2
            THEN 1
            ELSE 0
        END AS simbolo_con_transacciones_superiores_a_mercado
        FROM flagged_data GROUP BY simbolo_titulo, simbolo_tipo
) select simbolo_titulo, simbolo_tipo from data_agrupada_por_instrumento WHERE simbolo_con_transacciones_superiores_a_mercado=1

    

simbolo_titulo,simbolo_tipo
TSLA,Cedear
NVDA,Cedear
MELI,Cedear
GLOB,Cedear
NFLX,Cedear
MU,Cedear
V,Cedear
FSLR,Cedear
INTC,Cedear
DEO,Cedear


**¿Cuáles son los instrumentos con mayor desvio promedio entre precio operado y precio de mercado?**

In [0]:
%sql
WITH conjunto_a_analizar AS ( 
    -- Tomamos datos qué sean de los simbolos a analizar y qué tengan información de mercado
    SELECT * FROM iol_challenge.gold.fact_transaction 
        WHERE 
            simbolo_tipo IN ("Cedear", "Bono soberano") AND
            High IS NOT NULL AND Low is not null
), conjunto_con_desvios AS (
    SELECT 
        ((precio - valor_mercado_promedio) / valor_mercado_promedio)*100  as desvio_porcentual
        , *
        FROM conjunto_a_analizar
) SELECT
    simbolo_titulo, simbolo_tipo, 
    BROUND(AVG(desvio_porcentual), 2) AS desvio_porcentual_promedio
    FROM conjunto_con_desvios
        GROUP BY simbolo_titulo, simbolo_tipo
        ORDER BY AVG(desvio_porcentual) DESC LIMIT 10

simbolo_titulo,simbolo_tipo,desvio_porcentual_promedio
SPY,Cedear,200.74
STNE,Cedear,18.56
BBV,Cedear,3.13
BMA,Cedear,2.95
SCCO,Cedear,2.69
ACN,Cedear,2.53
BCS,Cedear,1.9
HSBC,Cedear,1.63
NEM,Cedear,1.2
HMY,Cedear,0.98


Databricks visualization. Run in Databricks to view.

Observamos qué el mayor desvio porcentual promedio esta en SPY, STNE y BBV.

**¿Cómo evolucionó la proporción de operaciones por canal mes a mes?**

In [0]:
%sql
SELECT 
    date_trunc('month', fecha) ::date
    ,origen
    ,SUM(total_transactions)
    FROM iol_challenge.gold.fact_transaction_daily 
    GROUP BY date_trunc('month', fecha), origen;

"date_trunc('month',fecha)::DATE",origen,SUM(total_transactions)
2026-01-01,App Mobile,32222
2026-01-01,Sitio Web Desktop,13452
2026-01-01,Sitio Web Responsive,1207
2026-01-01,API,539
2026-01-01,IOLnet,131
2026-02-01,Sitio Web Desktop,10480
2026-02-01,App Mobile,22916
2026-02-01,Sitio Web Responsive,991
2026-02-01,API,667
2026-02-01,IOLnet,109


Databricks visualization. Run in Databricks to view.

Revisamos cómo fue evolucionando la cantidad de operaciones por canal a lo largo de los tres meses con información disponible.

**¿De los clientes qué operaron en Enero, ¿Qué porcentaje volvio a operar en febrero y en marzo?**

In [0]:
%sql
WITH operaciones_por_periodo AS (
SELECT 
    id_cliente
    ,date_trunc('month', fecha)::date as periodo
    ,count(*) as operaciones
    FROM iol_challenge.gold.fact_transaction_daily 
        GROUP BY date_trunc('month', fecha)::date, id_cliente
), operacion_clientes_por_periodo AS (
    SELECT
        id_cliente
        ,CASE WHEN 
            SUM(CASE WHEN periodo = '2026-01-01' THEN 1 ELSE 0 END) > 0
            THEN 1
            ELSE 0
        END AS operaciones_enero
        ,CASE WHEN 
            SUM(CASE WHEN periodo = '2026-02-01' THEN 1 ELSE 0 END) > 0
            THEN 1
            ELSE 0
        END AS operaciones_febrero
        ,CASE WHEN 
            SUM(CASE WHEN periodo = '2026-03-01' THEN 1 ELSE 0 END) > 0
            THEN 1
            ELSE 0
        END AS operaciones_marzo
        FROM operaciones_por_periodo
            group by id_cliente
) SELECT 
    BROUND(SUM(CASE WHEN operaciones_enero = 1 AND operaciones_febrero = 1 THEN 1 ELSE 0 END) / COUNT(*) * 100 , 2) as clientes_que_volvieron_a_operar_en_febrero
    ,BROUND(SUM(CASE WHEN operaciones_enero = 1 AND operaciones_febrero = 1 and operaciones_marzo = 1 THEN 1 ELSE 0 END) / COUNT(*) * 100, 2) as clientes_que_volvieron_a_operar_en_febrero_y_marzo
 FROM operacion_clientes_por_periodo;

clientes_que_volvieron_a_operar_en_febrero,clientes_que_volvieron_a_operar_en_febrero_y_marzo
11.41,3.31


Observamos qué un 11% de los clientes qué operaron en enero volvieron a operar en febrero y un 3% volvieron a operar en febrero y además en Marzo.

**¿Existe correlación entre el canal de origen y la probabilidad de comprar por encima del mercado?**

In [0]:
%sql
WITH conjunto_a_analizar AS ( 
    -- Tomamos datos qué sean de los simbolos a analizar y qué tengan información de mercado
    SELECT * FROM iol_challenge.gold.fact_transaction 
        WHERE 
            simbolo_tipo IN ("Cedear", "Bono soberano") AND
            High IS NOT NULL AND Low is not null
), flagged_data AS (
    SELECT
        CASE WHEN valor_mercado_promedio > precio
            THEN 1
        ELSE 0
        END  AS transacion_superior_a_mercado
        ,*
    FROM conjunto_a_analizar
), data_agrupada_por_origen AS (
    select 
        origen
        ,BROUND(SUM(transacion_superior_a_mercado) / COUNT(*), 2) AS probablidad_comprar_superior_a_mercado
        FROM flagged_data GROUP BY origen
) select * from data_agrupada_por_origen ORDER BY probablidad_comprar_superior_a_mercado DESC;


origen,probablidad_comprar_superior_a_mercado
API,0.62
Sitio Web Responsive,0.5
Sitio Web Desktop,0.45
App Mobile,0.42
IOLnet,0.0


Databricks visualization. Run in Databricks to view.

Observamos qué hay una leve correlación entre la probabilidad de comprar a valores superiores a mercado y qué el canal de origen sea por API.

**Proponer y responder al menos una pregunta propia qué consideres valiosa para el negocio**

Pregunta propuesta de negocio a resolver:

**¿Qué transacciones de qué tipo operan con montos en ARS o en USD? ¿Se puede deducir el tipo de activo financiero en función de la moneda?**

In [0]:

%sql
SELECT 
    simbolo_tipo, moneda, COUNT(*)
 FROM iol_challenge.gold.dim_cotizaciones
    WHERE simbolo_tipo != 'A revisar'
    GROUP BY simbolo_tipo, moneda;


simbolo_tipo,moneda,COUNT(*)
Cedear,ARS,128
Accion local,ARS,43
Bono soberano,ARS,12
Bono soberano,USD,19


Databricks visualization. Run in Databricks to view.

Observamos qué los Cedear y las acciones locales se compran y venden en ARS. Por otro lado los bonos soberanos se comercian en USD y ARS. Esto quiere decir qué a priori solo la moneda qué se utiliza para la transaccion no se puede utilizar para deducir el tipo de activo financiero. Algunos bonos soberanos se compran en dolares y otros en pesos.